# 02 · Exploratory analysis
Metadata, coverage, missingness and a small coastal time series. Lightweight by design.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
from src import utils, data_access, preprocessing, visualisation
settings = utils.load_settings()
utils.set_seed(settings['project']['random_seed'])
%matplotlib inline

In [ ]:
ds = data_access.open_remote(settings)   # lazy
print(ds.attrs.get('source_citation'))
ds['thetao'].attrs

## Coverage (from coordinates only — no data transfer)

In [ ]:
import pandas as pd
t = pd.DatetimeIndex(ds.time.values)
print(t.min().date(), '→', t.max().date(), len(t), 'months')
print('lon', float(ds.longitude.min()), float(ds.longitude.max()), ds.sizes['longitude'])
print('lat', float(ds.latitude.min()), float(ds.latitude.max()), ds.sizes['latitude'])
print('depth', ds.depth.values)

## Small coastal box time series

In [ ]:
b = settings['sample_region']
box = data_access.open_remote(settings, min_lon=b['min_lon'], max_lon=b['max_lon'],
                              min_lat=b['min_lat'], max_lat=b['max_lat'])['thetao']
data_access.estimate_selection(box.to_dataset())

In [ ]:
box = data_access.safe_load(box, settings)
preprocessing.missing_report(preprocessing.surface(box))

In [ ]:
s = preprocessing.to_series(box)
preprocessing.describe_series(s)

In [ ]:
fig, ax = visualisation.plot_timeseries(s, f"Surface temperature, {b['name']}")
visualisation.save(fig, '02_box_timeseries')

In [ ]:
clim = preprocessing.monthly_climatology(s)
fig, ax = visualisation.plot_seasonality(clim, f"Monthly climatology, {b['name']}")
visualisation.save(fig, '02_box_seasonality')

In [ ]:
# One snapshot map of the study area (single month, ~0.1 MB)
snap = data_access.open_remote(settings, start='2020-01-01', end='2020-01-31')['thetao']
snap = data_access.safe_load(preprocessing.surface(snap).isel(time=0), settings)
fig, ax = visualisation.plot_map(snap, 'Surface potential temperature, Jan 2020')
visualisation.save(fig, '02_map_2020_01')